![Beyond Langfuse and LangSmith](assets/banner-04.svg)

Langfuse and LangSmith are popular, but they are not the only tools. Many companies already use a big monitoring platform such as **Datadog** or **Dynatrace** for their servers and databases, and want their AI traces in the same place.

Good news: you do not have to learn a new way of writing code for each tool. They all understand **OpenTelemetry**, the standard we used in notebook 01.

## The tool landscape

| | Tool | What it is | Good fit when |
|---|---|---|---|
| <img src="assets/logos/langfuse-color.png" width="36"> | **Langfuse** | Open source, built for AI apps | You want it free and on your own machines |
| <img src="assets/logos/langchain-color.png" width="36"> | **LangSmith** | Hosted, from the LangChain team | You use LangChain or LangGraph |
| <img src="assets/logos/phoenix-color.png" width="36"> | **Arize Phoenix** | Open source, built for AI apps | You want something light to run in notebooks |
| <img src="assets/logos/datadog-color.png" width="36"> | **Datadog** | Big monitoring platform with an LLM add-on | Your company already uses Datadog |
| <img src="assets/logos/dynatrace-color.png" width="36"> | **Dynatrace** | Big monitoring platform with AI features | Your company already uses Dynatrace |
| <img src="assets/logos/grafana-color.png" width="36"> | **Grafana** | Open source dashboards and trace storage | You already build dashboards in Grafana |

The first three are **made for AI apps**: they understand prompts, tokens and costs out of the box. The last three are **general monitoring platforms**: they put your AI traces next to everything else your company runs.

## One standard, many destinations

![How telemetry travels](assets/otel-pipeline.svg)

If you create your spans with OpenTelemetry, choosing a tool is just choosing **where to send them**. You change an address and a key, not your code.

Let's prove it. Below is the notebook 01 assistant, with plain OpenTelemetry `with` blocks. The only new part is the **exporter**: the piece that sends spans somewhere.

In [1]:
import base64
import os
import time

from dotenv import load_dotenv
from openai import OpenAI
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor

load_dotenv("../.env")

# Where to send the spans: here, our local Langfuse.
langfuse_key = base64.b64encode(
    f"{os.environ['LANGFUSE_PUBLIC_KEY']}:{os.environ['LANGFUSE_SECRET_KEY']}".encode()
).decode()
exporter = OTLPSpanExporter(
    endpoint="http://localhost:3000/api/public/otel/v1/traces",
    headers={"Authorization": f"Basic {langfuse_key}"},
)

provider = TracerProvider()
provider.add_span_processor(BatchSpanProcessor(exporter))
tracer = provider.get_tracer("weather-assistant")

In [2]:
client = OpenAI()
FAKE_WEATHER = {"Paris": "rainy, 14°C", "Tokyo": "sunny, 21°C", "London": "stormy, 11°C", "Dubai": "hot and dry, 35°C"}


def get_weather(city):
    time.sleep(0.5)
    return FAKE_WEATHER[city]


def ask_ai(question):
    reply = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": question}])
    return reply.choices[0].message.content


def weather_assistant(city):
    with tracer.start_as_current_span("weather_assistant (plain OpenTelemetry)"):
        with tracer.start_as_current_span("get_weather"):
            weather = get_weather(city)
        with tracer.start_as_current_span("ask_ai"):
            question = f"The weather in {city} is {weather}. Should I carry an umbrella? Answer in one short sentence."
            return ask_ai(question)


print(weather_assistant("London"))
provider.force_flush();  # send now

Yes, you should carry an umbrella.


Open Langfuse and you will find a trace called **`weather_assistant (plain OpenTelemetry)`**. We did not use any Langfuse code at all; Langfuse simply understands OpenTelemetry.

## Sending to a different tool

To send the exact same spans somewhere else, only the `endpoint` and `headers` change:

| Tool | `endpoint` | `headers` |
|---|---|---|
| Langfuse (local) | `http://localhost:3000/api/public/otel/v1/traces` | `Authorization: Basic <your keys>` |
| LangSmith | `https://api.smith.langchain.com/otel/v1/traces` | `x-api-key: <your key>` |
| Dynatrace | `https://<your-id>.live.dynatrace.com/api/v2/otlp/v1/traces` | `Authorization: Bearer <token>` |
| Datadog | Your Datadog site's OTLP address (see their docs) | `dd-api-key: <your key>` |

Your functions, your spans and your `with` blocks stay exactly the same.

## How to choose

| If... | Then try |
|---|---|
| You are learning or building a side project | **Langfuse**, on your laptop like today |
| Your app is built with LangChain or LangGraph | **LangSmith** |
| Your data must not leave your company | **Langfuse** or **Phoenix**, self-hosted |
| Your company already pays for Datadog or Dynatrace | Their LLM features, fed by OpenTelemetry |

## What you learned today

1. **Observability** means your app leaves enough information behind to explain what happened.
2. **Logs** are messages, **metrics** are numbers, **traces** are the story of one request, **spans** are its steps.
3. **Langfuse** and **LangSmith** add tracing with a few lines and record AI details for you: prompts, tokens, cost.
4. **OpenTelemetry** is the common standard underneath, so you can switch tools without rewriting code.

### Want to go further?

The [`advanced/`](advanced/) folder has deeper notebooks: OpenTelemetry internals, sessions and scores in Langfuse, latency and cost analytics, and a privacy filter that removes personal data before it leaves your network.